In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [4]:
from pyspark.sql import SparkSession

trade_df = spark.read.csv(
    "Files/data/processed/trades/trade_processed.csv",
    header=True,
    inferSchema=True
)

display(trade_df.limit(5))

df = spark.read.format("csv").option("header","true").load("Files/data/processed/trades/trade_processed.csv")
# df now is a Spark DataFrame containing CSV data from "Files/data/processed/trades/trade_processed.csv".
display(df)

StatementMeta(, 518f4f90-5701-4a98-946a-3ac91a065b3f, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa84d51f-9251-466e-85cc-6ce3e960ceff)

SynapseWidget(Synapse.DataFrame, c2cef41e-8d7c-45ec-801c-82c8f2d51104)

In [5]:
print(df.count())

StatementMeta(, 518f4f90-5701-4a98-946a-3ac91a065b3f, 7, Finished, Available, Finished, False)

21205192


**Trade Preprocessed**

In [2]:
from pyspark.sql import functions as F

df = spark.read.option(
    "header",
    "true"
).csv(
    "Files/data/processed/trades/trade_processed.csv",
    inferSchema=True
)

dup = (
    df.groupBy(
        "date",
        "symbol"
    )
    .count()
    .filter("count > 1")
)

print(
    dup.count()
)

StatementMeta(, 8df456bf-c704-4075-a596-3558017b14e0, 4, Finished, Available, Finished, False)

6624359


In [3]:
df.filter(
    df.symbol == "RELIANCE"
).show(20)

StatementMeta(, 8df456bf-c704-4075-a596-3558017b14e0, 5, Finished, Available, Finished, False)

+----------+--------+------------------+------------------+------------------+------------------+---------+------+
|      date|  symbol|              open|              high|               low|             close|   volume|source|
+----------+--------+------------------+------------------+------------------+------------------+---------+------+
|2001-01-01|RELIANCE|18.310957732669472|18.507532186048184| 18.25710104803247| 18.40520477294922| 42128577|   NSE|
|2001-01-01|RELIANCE|11.172913551330566|11.172913551330566|11.172913551330566|11.172913551330566| 15088640|   BSE|
|2001-01-02|RELIANCE| 18.32980708431301|19.334217418588832| 18.32442112815456| 19.08109474182129| 92764775|   NSE|
|2001-01-02|RELIANCE|11.660820960998535|11.660820960998535|11.660820960998535|11.660820960998535| 64415456|   BSE|
|2001-01-03|RELIANCE|19.118794052003363| 19.47962866848366|19.011083557390076|19.390766143798828|123856212|   NSE|
|2001-01-03|RELIANCE|11.739150047302246|11.739150047302246|11.739150047302246|11

In [4]:
print(df.select("symbol").distinct().count())

StatementMeta(, 8df456bf-c704-4075-a596-3558017b14e0, 6, Finished, Available, Finished, False)

6332


In [5]:
print(
    df.filter(df.source == "NSE")
      .select("symbol")
      .distinct()
      .count()
)

print(
    df.filter(df.source == "BSE")
      .select("symbol")
      .distinct()
      .count()
)

StatementMeta(, 8df456bf-c704-4075-a596-3558017b14e0, 7, Finished, Available, Finished, False)

3432
5516


In [6]:
df.printSchema()

StatementMeta(, 8df456bf-c704-4075-a596-3558017b14e0, 8, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- symbol: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- source: string (nullable = true)



In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read processed trade file
trade_df = spark.read.option(
    "header",
    "true"
).csv(
    "Files/data/processed/trades/trade_processed.csv",
    inferSchema=True
)

# NSE gets higher priority
trade_df = trade_df.withColumn(
    "priority",
    F.when(
        F.col("source") == "NSE",
        1
    ).otherwise(2)
)

# Keep NSE prices if available, otherwise BSE
window_spec = Window.partitionBy(
    "date",
    "symbol"
).orderBy(
    "priority"
)

price_df = (
    trade_df
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(
        F.col("rn") == 1
    )
    .select(
        "date",
        "symbol",
        "open",
        "high",
        "low",
        "close"
    )
)

# Sum volumes from NSE + BSE
volume_df = (
    trade_df
    .groupBy(
        "date",
        "symbol"
    )
    .agg(
        F.sum("volume")
        .alias("volume")
    )
)

# Final trade table
trade_final = (
    price_df
    .join(
        volume_df,
        ["date", "symbol"]
    )
)

# Save as Delta Table
trade_final.write.mode(
    "overwrite"
).format(
    "delta"
).saveAsTable(
    "trade_daily"
)

# Validation
print("SUCCESS")

print(
    "Rows:",
    trade_final.count()
)

display(
    trade_final.limit(10)
)

StatementMeta(, 8df456bf-c704-4075-a596-3558017b14e0, 9, Finished, Available, Finished, False)

SUCCESS
Rows: 14580830


SynapseWidget(Synapse.DataFrame, f173f592-1629-4933-a6be-c906c73f2e78)